# Twitch Analytics — Exploration de données historiques et actuelles

**Sources réelles :** [mingt2019/Twitch-Dataset](https://github.com/mingt2019/Twitch-Dataset) (snapshots datés de 2017) et API Twitch Helix (captures récentes).  
**But :** fouille, nettoyage, qualité, comparaison des périodes, faisabilité de la prévision.  
**Important :** deux périodes éloignées ne constituent pas une série temporelle continue ; les graphiques et prédictions les traitent séparément.

## 1. Lecture du CSV fusionné

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display

fichier = Path('twitch_dataset_complet.csv')
if not fichier.exists():
    raise FileNotFoundError("Place le CSV à côté du notebook, puis exécute collecte_twitch.py")
df = pd.read_csv(fichier, low_memory=False)
required = {'viewer_count','collected_at','source'}
assert required <= set(df.columns), f"Colonnes absentes : {required-set(df.columns)}"
df['viewer_count'] = pd.to_numeric(df['viewer_count'], errors='coerce')
df['collected_at'] = pd.to_datetime(df['collected_at'], utc=True, errors='coerce')
print('Nombre de lignes :',len(df))
print('Colonnes :',', '.join(df.columns))
display(df.head(5))

## 2. Fouille et qualité des données

In [ ]:
print('Lignes par source :')
display(df.groupby('source', dropna=False).agg(
    observations=('viewer_count','size'),
    audiences_valides=('viewer_count','count'),
    premiere=('collected_at','min'),
    derniere=('collected_at','max'),
    captures_distinctes=('collected_at','nunique')
))
print('Valeurs manquantes (%) :')
display((df.isna().mean()*100).round(1).rename('% manquants').to_frame())
print('Doublons stricts :',df.duplicated().sum())
print('Audiences négatives :',int((df['viewer_count']<0).sum()))
print('Dates invalides :',int(df['collected_at'].isna().sum()))

### Règles de nettoyage
On retire les audiences invalides et les dates illisibles. Les audiences élevées sont **conservées** : une grosse audience n'est pas automatiquement une erreur. Les doublons sont contrôlés par source, diffusion et instant. Les colonnes absentes des archives restent vides, sans inventer de valeur.

In [ ]:
clean = df.dropna(subset=['viewer_count','collected_at']).copy()
clean = clean[clean['viewer_count']>=0]
for c in ['id','user_id','game_name','user_name','language']:
    if c not in clean.columns:clean[c]=pd.NA
clean=clean.drop_duplicates(subset=['source','id','user_id','collected_at'])
print(f'Après nettoyage : {len(clean):,} lignes ({len(df)-len(clean):,} écartées)'.replace(',',' '))
fmt = FuncFormatter(lambda x,pos: f'{x:,.0f}'.replace(',',' '))

## 3. Graphique 1 — Distribution des audiences (zoom pour rester lisible)

In [ ]:
fig,ax=plt.subplots(figsize=(10,4))
for source,part in clean.groupby('source'):
    if len(part):
        seuil=part.viewer_count.quantile(.95)
        series=part.loc[part.viewer_count<=seuil,'viewer_count']
        ax.hist(series,bins=32,alpha=.5,label=f'{source} (95 % inférieurs)')
ax.set_xlabel('Spectateurs');ax.set_ylabel('Observations');ax.xaxis.set_major_formatter(fmt)
ax.set_title('Distribution des audiences par source — zoom au 95e percentile')
ax.legend();plt.tight_layout();plt.show()

**Lecture :** les écarts entre petites et grosses audiences sont importants. Le zoom sert seulement à la visualisation ; les lignes extrêmes ne sont pas supprimées du dataset.

## 4. Graphique 2 — Catégories les plus fréquentes

In [ ]:
fig,ax=plt.subplots(figsize=(10,4.5))
source=clean['source'].value_counts().index[0]
subset=clean[clean.source==source]
names=subset.game_name.fillna('Catégorie non renseignée').replace('', 'Catégorie non renseignée')
counts=names.value_counts().head(10).sort_values()
counts.plot.barh(ax=ax)
ax.set_title(f'10 catégories les plus représentées — {source}')
ax.set_xlabel('Nombre de relevés');ax.xaxis.set_major_formatter(fmt)
plt.tight_layout();plt.show()
print('Attention : si les archives n’ont pas de nom de jeu, cette figure montre leur indisponibilité.')

## 5. Graphique 3 — Évolution de l’audience observée (séparément par période)

In [ ]:
historic=clean[clean.source.str.contains('github',case=False,na=False)].copy()
if historic.collected_at.nunique()>=2:
    curve=historic.groupby('collected_at').viewer_count.median().sort_index()
    fig,ax=plt.subplots(figsize=(11,4))
    ax.plot(curve.index,curve.values,marker='o',markersize=3)
    ax.set_title('Médiane des audiences par capture — archives GitHub (2017)')
    ax.set_ylabel('Spectateurs (médiane)');ax.yaxis.set_major_formatter(fmt)
    ax.tick_params(axis='x',rotation=25);plt.tight_layout();plt.show()
else:
    print('Pas assez de captures historiques distinctes pour une évolution temporelle.')
print('Les captures Helix récentes ne sont pas reliées à 2017 par une courbe.')

## 6. Graphique 4 — Répartition selon les heures de collecte

In [ ]:
if historic.collected_at.nunique()>=2:
    h=historic.assign(heure=historic.collected_at.dt.hour).groupby('heure').viewer_count.median()
    fig,ax=plt.subplots(figsize=(9,4))
    ax.bar(h.index,h.values)
    ax.set_xticks(range(24));ax.set_xlabel('Heure UTC de collecte')
    ax.set_ylabel('Audience médiane');ax.yaxis.set_major_formatter(fmt)
    ax.set_title('Audience médiane selon l’heure — archive 2017')
    plt.tight_layout();plt.show()
else:print('Non disponible : historique multi-captures requis.')

## 7. Graphique 5 — Comparaison des distributions historiques / actuelles

In [ ]:
grouped=clean.groupby('source').viewer_count
stats=grouped.quantile([.25,.5,.75]).unstack().rename(columns={.25:'Q1',.5:'Médiane',.75:'Q3'})
display(stats.round().astype(int))
fig,ax=plt.subplots(figsize=(9,4))
y=range(len(stats));median=stats['Médiane'].to_numpy()
ax.errorbar(median,list(y),xerr=np.vstack([median-stats.Q1.to_numpy(),stats.Q3.to_numpy()-median]),fmt='o',capsize=4)
ax.set_yticks(list(y),stats.index);ax.set_xlabel('Spectateurs (médiane et quartiles)')
ax.xaxis.set_major_formatter(fmt);ax.set_title('Distributions par source — périodes non comparables directement')
plt.tight_layout();plt.show()

## 8. Bilan pour le projet fil rouge

- **Données disponibles :** snapshots publics GitHub de 2017 et données Helix actuelles ; la couverture exacte s’affiche dans la cellule 2.
- **Qualité :** vérifier manquants, doublons, horaires UTC et comptes de spectateurs anormaux.
- **Biais :** archives et top Helix ne suivent pas nécessairement les mêmes streamers ni les mêmes règles d’échantillonnage.
- **Architecture cible :** données brutes (Bronze) → nettoyage Python (Silver) → tables PostgreSQL (Gold) → dashboards Streamlit.
- **Machine Learning :** privilégier un premier modèle entraîné et testé **chronologiquement dans une seule période** ; ne pas utiliser 2017 comme si c’était la veille de 2026.
- **Limites :** quelques snapshots rapprochés suffisent pour tester les graphiques, mais pas pour généraliser à toutes les heures, tous les jours ou prévoir 2026. Une collecte Helix récurrente restera nécessaire.

**Conclusion orale :** « J’ai validé le principe d’un entrepôt de données multi-sources. J’ai identifié un historique réel, mais ancien, et une capture actuelle. Mon étape suivante sera de constituer un historique récent avec des relevés réguliers, puis de vérifier la qualité du modèle prédictif. »